# Análise do janelamento DNS

Fluxos iniciados na janela → agregação das features → Shannon → AttackRatio → Label.

Janelas sem novos fluxos não são criadas.

In [1]:
import pandas as pd

from src.config import (
    DATA_DIR,
    OUTPUT_DIR,
    WINDOW,
    ATTACK_RATIO_THRESHOLD,
    DNS_SAMPLE_ROWS,
    DNS_SAMPLE_START_ROW,
    ATTACK_INSERT_DURATION,
    BENIGN_SOURCE_FILES,
    ATTACK_FILES,
    SCENARIOS,
)

from src.data_loader import load_sample
from src.aggregation import rules_table
from src.windowing import create_windows
from src.baseline import build_benign_baseline
from src.attack_blocks import (
    build_attack_block,
    attack_block_info,
)
from src.scenarios import (
    prepare_scenario_attack_segments,
    build_scenario,
)

## 1. Amostra DNS

In [ ]:
dns_sample = load_sample(
    DATA_DIR / ATTACK_FILES["DNS"],
    DNS_SAMPLE_START_ROW,
    DNS_SAMPLE_ROWS,
    WINDOW,
)

display(
    dns_sample[
        [
            "FlowStart",
            "WindowStart",
            "CanonicalLabel",
        ]
    ].head(30)
)

## 2. Regras de agregação

In [ ]:
rules = rules_table(
    dns_sample
)

display(
    rules[
        rules["ExistsInCSV"]
    ].reset_index(
        drop=True
    )
)

## 3. Janelar a amostra DNS

In [ ]:
dns_windows = create_windows(
    dns_sample,
    ATTACK_RATIO_THRESHOLD,
)

display(
    dns_windows.head(30)
)

# Criação dos cenários

O baseline utiliza todas as janelas BENIGN disponíveis.

Para DNS, LDAP e SYN, o código mede primeiro a duração temporal real do bloco completo. A regra para as repetições é:

- se houver duração suficiente para todas as ocorrências de até 5 minutos, são usados blocos temporais distintos;
- se não houver duração suficiente, é criado um único bloco com duração `min(5 minutos, duração real disponível)` e esse bloco é repetido quantas vezes forem necessárias.

Assim, um ataque real de apenas 2 ou 3 minutos continua com sua duração original e pode reaparecer posteriormente no cenário.

## 4. Construir baseline benigno completo

In [10]:
baseline = build_benign_baseline(
    data_dir=DATA_DIR,
    source_files=BENIGN_SOURCE_FILES,
    output_path=(
        OUTPUT_DIR
        / "baseline"
        / "benign_baseline_1s.csv"
    ),
    window=WINDOW,
    attack_ratio_threshold=ATTACK_RATIO_THRESHOLD,
)

print(
    "Total de janelas BENIGN:",
    len(baseline),
)

print(
    baseline["Label"].value_counts()
)

Total de janelas BENIGN: 8075
Label
BENIGN    8075
Name: count, dtype: int64


## 5. Construir blocos completos DNS, LDAP e SYN

In [4]:
attack_blocks = {}
attack_block_summary = []

for attack, file_name in ATTACK_FILES.items():
    print(
        "Processando:",
        attack,
    )

    attack_blocks[
        attack
    ] = build_attack_block(
        csv_path=(
            DATA_DIR
            / file_name
        ),
        attack_name=attack,
        output_path=(
            OUTPUT_DIR
            / "attack_blocks"
            / f"{attack}_1s.csv"
        ),
        window=WINDOW,
        attack_ratio_threshold=ATTACK_RATIO_THRESHOLD,
    )

    attack_block_summary.append(
        attack_block_info(
            attack_blocks[
                attack
            ],
            attack,
        )
    )

attack_block_summary = pd.DataFrame(
    attack_block_summary
)

display(
    attack_block_summary
)

Processando: DNS
Processando: LDAP
Processando: SYN


,Attack,FirstTimestamp,LastTimestamp,RealDuration,TotalWindows,AttackWindows,BenignWindows
0,DNS,2018-12-01 10:51:39,2018-12-01 11:22:40,0 days 00:31:01,1624,1510,114
1,LDAP,2018-12-01 11:22:40,2018-12-01 11:32:32,0 days 00:09:52,593,593,0
2,SYN,2018-12-01 13:30:30,2018-12-01 13:34:27,0 days 00:03:57,238,238,0


## 6. Decidir automaticamente entre blocos distintos e repetição

A tabela mostra a duração disponível de cada ataque, quantas ocorrências os quatro cenários exigem e qual estratégia foi escolhida.

`distinct` significa blocos temporais diferentes.

`repeat` significa reutilização do mesmo bloco real.

In [5]:
attack_segments, segment_strategy = (
    prepare_scenario_attack_segments(
        attack_blocks=attack_blocks,
        scenarios=SCENARIOS,
        preferred_duration=ATTACK_INSERT_DURATION,
    )
)

display(
    segment_strategy
)

,Attack,RequiredOccurrences,AvailableDuration,PreferredDuration,SelectedBlockDuration,Strategy,SegmentsCreated
0,DNS,3,0 days 00:31:02,0 days 00:05:00,0 days 00:05:00,distinct,3
1,LDAP,2,0 days 00:09:53,0 days 00:05:00,0 days 00:05:00,repeat,2
2,SYN,2,0 days 00:03:58,0 days 00:05:00,0 days 00:03:58,repeat,2


## 7. Conferir os segmentos preparados

In [6]:
for attack, segments in attack_segments.items():
    print(
        "\n" + "=" * 70
    )

    print(
        attack
    )

    for index, segment in enumerate(
        segments,
        start=1,
    ):
        start = pd.to_datetime(
            segment[
                "OriginalTimestamp"
            ]
        ).min()

        end = pd.to_datetime(
            segment[
                "OriginalTimestamp"
            ]
        ).max()

        print(
            f"Ocorrência {index}:",
            start,
            "→",
            end,
            "| duração temporal:",
            (
                end
                - start
                + pd.Timedelta("1s")
            ),
            "| janelas:",
            len(segment),
        )


DNS
Ocorrência 1: 2018-12-01 10:51:39 → 2018-12-01 10:56:38 | duração temporal: 0 days 00:05:00 | janelas: 281
Ocorrência 2: 2018-12-01 10:56:39 → 2018-12-01 11:01:38 | duração temporal: 0 days 00:05:00 | janelas: 300
Ocorrência 3: 2018-12-01 11:01:39 → 2018-12-01 11:06:38 | duração temporal: 0 days 00:05:00 | janelas: 265

LDAP
Ocorrência 1: 2018-12-01 11:22:40 → 2018-12-01 11:27:39 | duração temporal: 0 days 00:05:00 | janelas: 300
Ocorrência 2: 2018-12-01 11:22:40 → 2018-12-01 11:27:39 | duração temporal: 0 days 00:05:00 | janelas: 300

SYN
Ocorrência 1: 2018-12-01 13:30:30 → 2018-12-01 13:34:27 | duração temporal: 0 days 00:03:58 | janelas: 238
Ocorrência 2: 2018-12-01 13:30:30 → 2018-12-01 13:34:27 | duração temporal: 0 days 00:03:58 | janelas: 238


C:\Users\Leandro Tosta\AppData\Local\Temp\ipykernel_43588\4191046235.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  + pd.Timedelta("1s")
C:\Users\Leandro Tosta\AppData\Local\Temp\ipykernel_43588\4191046235.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  + pd.Timedelta("1s")
C:\Users\Leandro Tosta\AppData\Local\Temp\ipykernel_43588\4191046235.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  + pd.Timedelta("1s")
C:\Users\Leandro Tosta\AppData\Local\Temp\ipykernel_43588\419104

## 8. Gerar os quatro cenários

In [11]:
scenario_results = {}

for scenario_name, sequence in SCENARIOS.items():
    scenario = build_scenario(
        scenario_name=scenario_name,
        sequence=sequence,
        baseline=baseline,
        attack_segments=attack_segments,
        output_path=(
            OUTPUT_DIR
            / "scenarios"
            / f"{scenario_name}_1s.csv"
        ),
        window=WINDOW,
    )

    scenario_results[
        scenario_name
    ] = scenario

    print(
        "\n",
        scenario_name,
    )

    print(
        "Janelas:",
        len(scenario),
    )

    print(
        scenario[
            "Label"
        ].value_counts()
    )


 Consistency
Janelas: 8921
Label
BENIGN    8122
DNS        799
Name: count, dtype: int64

 Generalization
Janelas: 8956
Label
BENIGN    8095
DNS        561
LDAP       300
Name: count, dtype: int64

 Adaptation
Janelas: 8894
Label
BENIGN    8095
DNS        561
SYN        238
Name: count, dtype: int64

 Recurrence
Janelas: 9732
Label
BENIGN    8095
LDAP       600
DNS        561
SYN        476
Name: count, dtype: int64


## 9. Resumo final dos cenários

In [ ]:
summary = []

for scenario_name, scenario in scenario_results.items():
    summary.append({
        "Scenario": scenario_name,
        "Windows": len(
            scenario
        ),
        "BenignWindows": int(
            (
                scenario[
                    "Label"
                ]
                == "BENIGN"
            ).sum()
        ),
        "AttackWindows": int(
            (
                scenario[
                    "Label"
                ]
                != "BENIGN"
            ).sum()
        ),
        "AttackRatioPercent": (
            (
                scenario[
                    "Label"
                ]
                != "BENIGN"
            ).mean()
            * 100
        ),
    })

display(
    pd.DataFrame(
        summary
    )
)